In [1]:
import os
import shutil
from sklearn.model_selection import train_test_split

In [2]:
dataset_path = "dataset"
classes = ["Cat", "Dog"]

In [3]:
for class_name in classes:
    class_path = os.path.join(dataset_path, class_name)
    images = os.listdir(class_path)

    train_imgs, val_imgs = train_test_split(images, test_size=0.15, random_state=42)

    # Create directories
    os.makedirs(f"{dataset_path}/train/{class_name}", exist_ok=True)
    os.makedirs(f"{dataset_path}/val/{class_name}", exist_ok=True)

    # Copy images
    for img in train_imgs:
        shutil.copy2(f"{class_path}/{img}", f"{dataset_path}/train/{class_name}/")
    for img in val_imgs:
        shutil.copy2(f"{class_path}/{img}", f"{dataset_path}/val/{class_name}/")

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'dataset\\Cat'

In [4]:
from ultralytics import YOLO

model = YOLO("yolo11n-cls") # this model is perfect for binary problems

In [5]:
results = model.train(
    data="dataset/",     # by default , yolo detects the val and train folders
    epochs=10,
    imgsz=224,
    batch=32,
    workers=10,
    project="cat_dog_cls",
    name="yolo11",
    augment=False,
)

New https://pypi.org/project/ultralytics/8.3.235 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.228  Python-3.13.5 torch-2.9.1+cpu CPU (13th Gen Intel Core i7-13650HX)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11, nbs=64, nms=False, opset=None, optimize=F

c:\Users\bello\AppData\Local\Programs\Python\Python313\Lib\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


train: New cache created: C:\Users\bello\Desktop\projet douik\YOLO\dataset\train.cache
val: Fast image access  (ping: 0.10.0 ms, read: 70.223.5 MB/s, size: 25.8 KB)
val: Scanning C:\Users\bello\Desktop\projet douik\YOLO\dataset\val... 3742 images, 8 corrupt: 100% ━━━━━━━━━━━━ 3750/3750 6.4Kit/s 0.6s0.1s
val: C:\Users\bello\Desktop\projet douik\YOLO\dataset\val\Cat\11874.jpg: ignoring corrupt image/label: Invalid image format GIF. Supported formats are:
images: {'jpg', 'tif', 'mpo', 'jpeg', 'dng', 'pfm', 'tiff', 'png', 'webp', 'bmp', 'heic'}
videos: {'wmv', 'mpg', 'mp4', 'mkv', 'mpeg', 'asf', 'mov', 'ts', 'm4v', 'avi', 'webm', 'gif'}
val: C:\Users\bello\Desktop\projet douik\YOLO\dataset\val\Cat\9565.jpg: ignoring corrupt image/label: Invalid image format GIF. Supported formats are:
images: {'jpg', 'tif', 'mpo', 'jpeg', 'dng', 'pfm', 'tiff', 'png', 'webp', 'bmp', 'heic'}
videos: {'wmv', 'mpg', 'mp4', 'mkv', 'mpeg', 'asf', 'mov', 'ts', 'm4v', 'avi', 'webm', 'gif'}
val: C:\Users\bello\Desk

In [1]:
import os
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# ===========================
# CONFIGURATION GÉNÉRALE
# ===========================
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 10)

dataset_path = "dataset"
classes = ["Cat", "Dog"]

# ===========================
# 1. PAS DE SCAN DU DATASET
# ===========================
print("\n" + "="*70)
print("📁 DATASET UTILISÉ (SANS SCAN DES IMAGES)")
print("="*70)

print("""
✔ Dataset déjà structuré:
dataset/
 ├── train/Cat
 ├── train/Dog
 ├── val/Cat
 ├── val/Dog

✔ Aucune lecture des fichiers images
✔ Aucune consommation disque inutile
""")

# ===========================
# 2. ENTRAÎNEMENT DU MODÈLE
# ===========================
print("\n" + "="*70)
print("🚀 ENTRAÎNEMENT DU MODÈLE")
print("="*70 + "\n")

model = YOLO("yolo11n-cls")

results = model.train(
    data="dataset/",
    epochs=10,
    imgsz=224,
    batch=32,
    workers=6,
    project="cat_dog_cls",
    name="yolo11",
    augment=False,
    plots=True
)

print("\n✅ Entraînement terminé!")

# ===========================
# 3. ANALYSE DES RÉSULTATS
# ===========================
print("\n" + "="*70)
print("📊 ANALYSE DES RÉSULTATS")
print("="*70 + "\n")

results_path = Path("cat_dog_cls/yolo11")
results_csv = results_path / "results.csv"

if not results_csv.exists():
    raise FileNotFoundError("❌ results.csv introuvable !")

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

# Détection automatique des colonnes
loss_col = next((c for c in df.columns if "loss" in c.lower()), None)
acc_col = next((c for c in df.columns if "top1" in c.lower()), None)
acc5_col = next((c for c in df.columns if "top5" in c.lower()), None)
lr_col = next((c for c in df.columns if "lr" in c.lower()), None)

# ===========================
# 4. DASHBOARD PRINCIPAL
# ===========================
fig = plt.figure(figsize=(20, 12))

# LOSS
ax1 = plt.subplot(2, 3, 1)
ax1.plot(df["epoch"], df[loss_col], marker="o")
ax1.set_title("Training Loss")

# ACC TOP-1
ax2 = plt.subplot(2, 3, 2)
ax2.plot(df["epoch"], df[acc_col]*100, marker="o")
ax2.set_ylim(0, 100)
ax2.set_title("Top-1 Accuracy")

# ACC TOP-5
ax3 = plt.subplot(2, 3, 3)
if acc5_col:
    ax3.plot(df["epoch"], df[acc5_col]*100, marker="s")
ax3.set_ylim(0, 100)
ax3.set_title("Top-5 Accuracy")

# LR
ax4 = plt.subplot(2, 3, 4)
if lr_col:
    ax4.plot(df["epoch"], df[lr_col])
ax4.set_yscale("log")
ax4.set_title("Learning Rate")

# LOSS VS ACC
ax5 = plt.subplot(2, 3, 5)
ax5.plot(df["epoch"], df[loss_col], label="Loss")
ax5_t = ax5.twinx()
ax5_t.plot(df["epoch"], df[acc_col]*100, color="green", label="Accuracy")
ax5.set_title("Loss vs Accuracy")

# SUMMARY
ax6 = plt.subplot(2, 3, 6)
ax6.axis("off")

final = df.iloc[-1]
best = df.iloc[df[acc_col].idxmax()]

summary = f"""
RÉSUMÉ FINAL

Epochs: {len(df)}

Meilleure Accuracy:
Epoch {int(best['epoch'])}
Accuracy: {best[acc_col]*100:.2f}%
Loss: {best[loss_col]:.4f}

Dernière Epoch:
Accuracy: {final[acc_col]*100:.2f}%
Loss: {final[loss_col]:.4f}
"""

ax6.text(0.05, 0.5, summary, fontsize=12, family="monospace")

plt.tight_layout()
plt.savefig("cat_dog_cls/yolo11/training_dashboard.png", dpi=300)
plt.show()

# ===========================
# 5. VALIDATION
# ===========================
print("\n" + "="*70)
print("✅ VALIDATION")
print("="*70)

best_model = YOLO("cat_dog_cls/yolo11/weights/best.pt")
val_results = best_model.val(data="dataset/")

print(f"Top-1 Accuracy: {val_results.top1:.4f}")
print(f"Top-5 Accuracy: {val_results.top5:.4f}")

# ===========================
# 6. TEST DE PRÉDICTION
# ===========================
print("\n" + "="*70)
print("🔍 TESTS DE PRÉDICTION")
print("="*70)

test_images = ["test_cat.jpg", "test_dog.jpg"]

for img in test_images:
    if os.path.exists(img):
        r = best_model.predict(img, verbose=False)[0]
        pred = "Cat" if r.probs.top1 == 0 else "Dog"
        conf = r.probs.top1conf.item()
        print(f"{img} → {pred} ({conf*100:.2f}%)")
    else:
        print(f"{img} non trouvé")

# ===========================
# 7. FIN
# ===========================
print("\n" + "="*70)
print("✅ SCRIPT TERMINÉ SANS AUCUN SCAN D'IMAGES")
print("="*70)



📁 DATASET UTILISÉ (SANS SCAN DES IMAGES)

✔ Dataset déjà structuré:
dataset/
 ├── train/Cat
 ├── train/Dog
 ├── val/Cat
 ├── val/Dog

✔ Aucune lecture des fichiers images
✔ Aucune consommation disque inutile


🚀 ENTRAÎNEMENT DU MODÈLE

New https://pypi.org/project/ultralytics/8.3.235 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.228  Python-3.13.5 torch-2.9.1+cpu CPU (13th Gen Intel Core i7-13650HX)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0

KeyboardInterrupt: 